# A cost-efficient Deep Learning model to emulate surface mass balance (SMB) over the Antartic Peninsula

In [22]:
from datetime import datetime
from pathlib import Path

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from download_data import download_data
from super_resolution import rcm_emulator_loc_entrypoint

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Select an input date:

In [2]:
DATE = datetime(2023, 1, 1)

Select an input location:

In [3]:
LOCATION = (-75.0076, -65.1279)

Here I download the data with ddm, but API should do it internally.

Download of:
1. ERA5_MONTHLY
2. RACMO

In [28]:
from download_data import download_era5_from_ddm, download_rcm_from_ddm
from datetime import timedelta

era5_path = download_era5_from_ddm(
    start_date=DATE,
    end_date=DATE + timedelta(days=31),
    output_dir=Path("local_data/ERA5"),
)

racmo_path = download_rcm_from_ddm(
    start_date=DATE,
    end_date=DATE + timedelta(days=31),
    output_dir=Path("local_data/RACMO")
)

[    INFO 2026-09-15 CEST 17:31:44.113 pid  2081,                        main.py, ln  98, in             download_data()] Processing data download request dataset_id='ERA5_REANALYSIS_MONTHLY_MEANS' output_path=None output_parent_path=PosixPath('local_data/ERA5/ERA5') start_time=datetime.datetime(2023, 1, 1, 0, 0) end_time=datetime.datetime(2023, 2, 1, 0, 0) epsg4326_polygon=<POLYGON ((180 -90, 180 -60, -180 -60, -180 -90, 180 -90))> variables_to_download=['d2m', 'msl', 'sst', 'sp', 't2m', 'tp', 'u10', 'v10'] mask_data_to_polygon=False raw_download=False
[2026-09-15 17:31:44,113] INFO in data_download_module.main: Processing data download request dataset_id='ERA5_REANALYSIS_MONTHLY_MEANS' output_path=None output_parent_path=PosixPath('local_data/ERA5/ERA5') start_time=datetime.datetime(2023, 1, 1, 0, 0) end_time=datetime.datetime(2023, 2, 1, 0, 0) epsg4326_polygon=<POLYGON ((180 -90, 180 -60, -180 -60, -180 -90, 180 -90))> variables_to_download=['d2m', 'msl', 'sst', 'sp', 't2m', 'tp', '

### Call the entrypoint

In [42]:
# it will take 20mins without GPU
OUTPUT_PATH = './tmp/predictions.zarr'
rcm_emulator_loc_entrypoint.main(
    gcm_zarr_path=era5_path,
    rcm_zarr_path=racmo_path,
    location=LOCATION,
    output_zarr_path=OUTPUT_PATH,
    model_name="model_attention_unet")

[    INFO 2026-09-15 CEST 17:43:43.978 pid  2081,                reprojection.py, ln 145, in reproject_era5_to_racmo_grid()] Reprojecting  ERA5 to RACMO grid
[2026-09-15 17:43:43,978] INFO in super_resolution.rcm_emulator.pre_processing_module.reprojection: Reprojecting  ERA5 to RACMO grid
[    INFO 2026-09-15 CEST 17:43:44.020 pid  2081,                reprojection.py, ln 159, in reproject_era5_to_racmo_grid()] Interpolating ['d2m', 'msl', 'sp', 'spatial_ref', 'sst', 't2m', 'tp', 'u10', 'v10'] onto 563x295 grid...
[2026-09-15 17:43:44,020] INFO in super_resolution.rcm_emulator.pre_processing_module.reprojection: Interpolating ['d2m', 'msl', 'sp', 'spatial_ref', 'sst', 't2m', 'tp', 'u10', 'v10'] onto 563x295 grid...
Month 2/2: 100%|██████████| 1/1 [00:01<00:00,  1.49s/it]
/home/ngaillard/miniconda3/envs/dtc/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=3, endianness='little')) does not have a Zarr V3

In [43]:
import pandas as pd

preds_ds =xr.open_zarr(OUTPUT_PATH, consolidated=True)

In [44]:
preds_ds

<xarray.Dataset> Size: 639kB
Dimensions:         (time: 2, y: 224, x: 224, time_r: 1, channels: 8,
                     y_era5: 16, x_era5: 32)
Coordinates:
  * time            (time) datetime64[ns] 16B 2023-01-01 2023-02-01
  * y               (y) float64 2kB 4.642e+05 4.662e+05 ... 9.082e+05 9.102e+05
  * x               (x) float64 2kB -1.711e+06 -1.709e+06 ... -1.265e+06
  * time_r          (time_r) datetime64[ns] 8B 2023-01-16
  * channels        (channels) <U3 96B 'd2m' 'msl' 'sp' ... 'tp' 'u10' 'v10'
  * y_era5          (y_era5) float64 128B 4.514e+05 4.794e+05 ... 8.714e+05
  * x_era5          (x_era5) float64 256B -1.713e+06 -1.699e+06 ... -1.279e+06
Data variables:
    predictions     (time, y, x) float32 401kB dask.array<chunksize=(1, 224, 224), meta=np.ndarray>
    target_patches  (time_r, y, x) float32 201kB dask.array<chunksize=(1, 224, 224), meta=np.ndarray>
    x_cropped       (time, channels, y_era5, x_era5) float32 33kB dask.array<chunksize=(2, 8, 16, 32), meta=np.ndarray>
Attributes:
    lon:            -65.1279
    lat:            -75.0076
    x_3031:         -1486022.709359959
    y_3031:         688909.4451180231
    era5_y_slice:   {'start': 148, 'stop': 196, 'step': None}
    era5_x_slice:   {'start': 152, 'stop': 200, 'step': None}
    racmo_y_slice:  {'start': 2306, 'stop': 2530, 'step': None}
    racmo_x_slice:  {'start': 1122, 'stop': 1346, 'step': None}

### Plot timeseries

In [ ]:
predictions = preds_ds['predictions'].isel(time=0).values

In [ ]:
vmin = np.min([np.min(predictions), np.min(target_tiles)])
vmax = np.max([np.max(predictions), np.max(target_tiles)])

In [ ]:
y_vals = np.mean(predictions, axis=(1, 2))
y_vals_center = predictions[:, predictions.shape[1]//2, predictions.shape[2]//2]

era5_vars = ["d2m", "msl", "sp", "sst", "t2m", "tp", "u10", "v10"]
y_precip = np.mean(input_tiles[:, 0, 5, :, :], axis=(1, 2))  
y_temp = np.mean(input_tiles[:, 0, 4, :, :], axis=(1, 2))

timeline = era5_xy.time.values

<xarray.Dataset> Size: 204kB
Dimensions:      (y: 224, x: 224)
Coordinates:
  * y            (y) float64 2kB 4.642e+05 4.662e+05 ... 9.082e+05 9.102e+05
  * x            (x) float64 2kB -1.711e+06 -1.709e+06 ... -1.267e+06 -1.265e+06
    time         datetime64[ns] 8B 2023-01-01
Data variables:
    predictions  (y, x) float32 201kB dask.array<chunksize=(224, 224), meta=np.ndarray>
Attributes:
    lon:            -65.1279
    lat:            -75.0076
    x_3031:         -1486022.709359959
    y_3031:         688909.4451180231
    era5_y_slice:   {'start': 148, 'stop': 196, 'step': None}
    era5_x_slice:   {'start': 152, 'stop': 200, 'step': None}
    racmo_y_slice:  {'start': 2306, 'stop': 2530, 'step': None}
    racmo_x_slice:  {'start': 1122, 'stop': 1346, 'step': None}

In [16]:
df = ds.mean(dim=["x", "y"]).to_dataframe()

/home/ngaillard/miniconda3/envs/dtc/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


ValueError: no valid index for a 0-dimensional object

In [ ]:
df